In [1]:
import os
import sys

import torch

sys.path.append('../..')

from fnofound.agents.agent import NeuralOpSystemEnvironment, InitialConditionsAgent

In [2]:
env = NeuralOpSystemEnvironment()

tensor_1 = torch.rand(size=(10, 1, 12, 32, 32))
tensor_2 = torch.rand(size=(10, 2, 12, 32, 32))
tensor_3 = torch.rand(size=(10, 1, 12, 32, 32))

initial_cond_loader = InitialConditionsAgent(env = env,
                                             initial_conds = {'1': tensor_1,
                                                              '2': tensor_2,
                                                              '3': tensor_3},
                                             additional_channels = torch.rand(size=(10, 3, 12, 32, 32)), 
                                             )

In [4]:
initial_cond_loader.formInputTensor().shape

torch.Size([10, 7, 12, 32, 32])

In [8]:
import torch
import argparse
from datetime import datetime

from typing import List

import h5py
import glob
import sys
import os

sys.path.append('.')

from gpn_mamba import Heatmap

In [24]:
b_max = 100

filepaths = ['/media/mikemaslyaev/Data/GPN_data/gas_horizontal/dataset_gas_horizontal_1.hdf5',
                '/media/mikemaslyaev/Data/GPN_data/gas_horizontal/dataset_gas_horizontal_2.hdf5',
                '/media/mikemaslyaev/Data/GPN_data/gas_horizontal/dataset_gas_horizontal_3.hdf5']

for idx, filepath_new in enumerate(filepaths):
    if idx == 0:
        with h5py.File(filepath_new, "r") as f:
            raw_solutions = torch.tensor(f['dataset'][:b_max, :b_max], dtype=torch.float32)  # [T, N, 2, H, W] 1100
            print(f['source'][:b_max, :b_max].shape)
            raw_forcings  = torch.tensor(f['source'][:b_max, :b_max], dtype=torch.float32)[:, :, :-1, ...]   # [T, N, 1, H, W]
            raw_forcings = raw_forcings.permute(2, 0, 1, 3, 4)
    else:
        with h5py.File(filepath_new, "r") as f:
            raw_solutions = torch.cat([raw_solutions, torch.tensor(f['dataset'][:b_max, :b_max], dtype=torch.float32)], dim = 1)  # [T, N, 2, H, W] 1100
            print(f['source'][:b_max, :b_max].shape)

            raw_forcings = torch.cat([raw_forcings, 
                                        torch.tensor(f['source'][:b_max, :b_max], dtype=torch.float32)[:, :, :-1, ...].permute(2, 0, 1, 3, 4)], dim = 1)

print('shapes: ', raw_solutions.shape, raw_forcings.shape)

(100, 1, 21, 113, 134)
(100, 1, 21, 113, 134)
(100, 1, 21, 113, 134)
shapes:  torch.Size([21, 300, 4, 113, 134]) torch.Size([20, 300, 1, 113, 134])


In [25]:
N = raw_solutions.shape[1]
perm = torch.randperm(N)
raw_solutions = raw_solutions[1:, perm, ...]
raw_forcings = raw_forcings[:, perm, ...]

# To [N, T, C, H, W]
solutions = raw_solutions.permute(1, 2, 0, 3, 4)
forcings = raw_forcings.permute(1, 2, 0, 3, 4)

In [26]:
print('shapes: ', solutions.shape, forcings.shape)

shapes:  torch.Size([300, 4, 20, 113, 134]) torch.Size([300, 1, 20, 113, 134])
